# EvalPlus benchmark: Qwen2.5-Coder-7B (base) vs v3 (merged)

HumanEval + HumanEval+ (EvalPlus), vLLM, greedy pass@1.

**Trước khi chạy (Kaggle):**
1. Settings → Accelerator: **GPU T4 x2**. Settings → Internet: **On**.
2. Add-ons → Secrets: thêm `HF_TOKEN` (quyền đọc, repo v3-merged đang private) và attach vào notebook.

Model 7B fp16 (~15GB) không vừa 1 T4 nên dùng `--tp 2`; T4 không có bf16 nên dùng `float16`.


## 1. Cài đặt

In [1]:
!pip install -q uv
!uv venv /kaggle/working/vllm_env --python 3.12
!uv pip install -q --python /kaggle/working/vllm_env/bin/python "evalplus[vllm]"
!/kaggle/working/vllm_env/bin/python -c "import torch, vllm; print('torch', torch.__version__, 'cuda', torch.version.cuda, '| vllm', vllm.__version__)"

Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: vllm_env
Activate with: source vllm_env/bin/activate
torch 2.13.0+cu130 cuda 13.0 | vllm 0.32.0


## 2. Config

In [2]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

BASE = "Qwen/Qwen2.5-Coder-7B"
V3   = "hoangnh39/qwen2.5-coder-7b-opencode-hq10k-v3-merged"
TP, DTYPE = 2, "float16"
DATASETS = ["mbpp"]   # báo cả MBPP và MBPP+; thêm "humaneval" nếu muốn chạy lại
EVAL = "/kaggle/working/vllm_env/bin/evalplus.evaluate"

!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


## 3. Base (completion prompt)

In [3]:
for ds in DATASETS:
    print(f"===== base / {ds} =====")
    !{EVAL} --model {BASE} --dataset {ds} --backend vllm --greedy --force-base-prompt --tp {TP} --dtype {DTYPE} 2>&1 | tee evalplus_base_{ds}.log | tail -n 40

===== base / mbpp =====
Codegen: Mbpp/794 @ Qwen/Qwen2.5-Coder-7B
Codegen: Mbpp/796 @ Qwen/Qwen2.5-Coder-7B
Codegen: Mbpp/797 @ Qwen/Qwen2.5-Coder-7B
Codegen: Mbpp/798 @ Qwen/Qwen2.5-Coder-7B
Codegen: Mbpp/799 @ Qwen/Qwen2.5-Coder-7B
Codegen: Mbpp/800 @ Qwen/Qwen2.5-Coder-7B
Codegen: Mbpp/801 @ Qwen/Qwen2.5-Coder-7B
Codegen: Mbpp/803 @ Qwen/Qwen2.5-Coder-7B
Codegen: Mbpp/804 @ Qwen/Qwen2.5-Coder-7B
Codegen: Mbpp/805 @ Qwen/Qwen2.5-Coder-7B
Codegen: Mbpp/806 @ Qwen/Qwen2.5-Coder-7B
Codegen: Mbpp/807 @ Qwen/Qwen2.5-Coder-7B
Codegen: Mbpp/808 @ Qwen/Qwen2.5-Coder-7B
Codegen: Mbpp/809 @ Qwen/Qwen2.5-Coder-7B
mbpp •100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378/378 • 0:21:24
(MainProcess pid=152) INFO 10-23 09:46:38 [utils.py:628] [shutdown] Process manager: send sigterm to process EngineCore
(EngineCore pid=199) INFO 10-23 09:46:38 [core.py:1454] [shutdown] EngineCore: trigger received signal=SIGTERM
(EngineCore pid=199) INFO 10-23 09:46:38 [core.py:1635] [shutdown] EngineCore: start mo

## 4. v3 (chat template)

In [4]:
for ds in DATASETS:
    print(f"===== v3 / {ds} =====")
    !{EVAL} --model {V3} --dataset {ds} --backend vllm --greedy --tp {TP} --dtype {DTYPE} 2>&1 | tee evalplus_v3_{ds}.log | tail -n 40

===== v3 / mbpp =====
Codegen: Mbpp/788 @ hoangnh39/qwen2.5-coder-7b-opencode-hq10k-v3-merged
Codegen: Mbpp/790 @ hoangnh39/qwen2.5-coder-7b-opencode-hq10k-v3-merged
Codegen: Mbpp/791 @ hoangnh39/qwen2.5-coder-7b-opencode-hq10k-v3-merged
Codegen: Mbpp/792 @ hoangnh39/qwen2.5-coder-7b-opencode-hq10k-v3-merged
Codegen: Mbpp/793 @ hoangnh39/qwen2.5-coder-7b-opencode-hq10k-v3-merged
Codegen: Mbpp/794 @ hoangnh39/qwen2.5-coder-7b-opencode-hq10k-v3-merged
Codegen: Mbpp/796 @ hoangnh39/qwen2.5-coder-7b-opencode-hq10k-v3-merged
Codegen: Mbpp/797 @ hoangnh39/qwen2.5-coder-7b-opencode-hq10k-v3-merged
Codegen: Mbpp/798 @ hoangnh39/qwen2.5-coder-7b-opencode-hq10k-v3-merged
Codegen: Mbpp/799 @ hoangnh39/qwen2.5-coder-7b-opencode-hq10k-v3-merged
Codegen: Mbpp/800 @ hoangnh39/qwen2.5-coder-7b-opencode-hq10k-v3-merged
Codegen: Mbpp/801 @ hoangnh39/qwen2.5-coder-7b-opencode-hq10k-v3-merged
Codegen: Mbpp/803 @ hoangnh39/qwen2.5-coder-7b-opencode-hq10k-v3-merged
Codegen: Mbpp/804 @ hoangnh39/qwen2.5-code

## 5. Summary

In [5]:
!grep -E "pass@1|mbpp|humaneval" evalplus_*_*.log

evalplus_base_mbpp.log:Sanitized code outputs will be saved to evalplus_results/mbpp/Qwen--Qwen2.5-Coder-7B_vllm_temp_0.0.jsonl
evalplus_base_mbpp.log:Raw outputs will be saved to evalplus_results/mbpp/Qwen--Qwen2.5-Coder-7B_vllm_temp_0.0.raw.jsonl
evalplus_base_mbpp.log:Downloading dataset from https://github.com/evalplus/mbppplus_release/releases/download/v0.2.0/MbppPlus.jsonl.gz
evalplus_base_mbpp.log:mbpp •100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378/378 • 0:21:24
evalplus_base_mbpp.log:/kaggle/working/vllm_env/lib/python3.12/site-packages/evalplus/evaluate.py:240: UserWarning: 1 samples to be tested: {'Mbpp/599 (line 269 in evalplus_results/mbpp/Qwen--Qwen2.5-Coder-7B_vllm_temp_0.0.jsonl)'}
evalplus_base_mbpp.log:mbpp (base tests)
evalplus_base_mbpp.log:pass@1:	0.786
evalplus_base_mbpp.log:mbpp+ (base + extra tests)
evalplus_base_mbpp.log:pass@1:	0.640
evalplus_v3_mbpp.log:Sanitized code outputs will be saved to evalplus_results/mbpp/hoangnh39--qwen2.5-coder-7b-opencode-hq10k

## Nếu lỗi

- `CUDA out of memory`: thêm `--gpu-memory-utilization 0.85` hoặc giảm `--max-model-len`, hoặc chạy lại sau khi Restart Session.
- vLLM không khởi động được trên T4 (lỗi attention backend): thử `os.environ["VLLM_ATTENTION_BACKEND"] = "XFORMERS"` trong cell Config.
- Cell cài đặt lỗi (uv không tải được Python 3.12): đổi `--python 3.12` thành `--python 3.13` hoặc bỏ tham số.
- Điểm v3 thấp bất thường: chat template của Qwen tự thêm system prompt mặc định, trong khi lúc train không có.
